# Cross Entropy（交叉熵损失）

交叉熵衡量模型预测的类别分布与真实分布之间的差异，是多分类任务最常用的训练目标。通常模型先输出 logits，再与 Softmax 联合计算，以获得更好的数值稳定性。

对单个样本，真实标签的 one-hot 分布为 $y$，预测概率为 $p$，交叉熵为：

$$\mathcal{L}=-\sum_{c=1}^{C}y_c\log p_c.$$

若真实类别索引为 $t$，则可简化为：

$$\mathcal{L}=-\log p_t.$$

令 logits 为 $z$，且 $p_c=\frac{e^{z_c}}{\sum_{j=1}^{C}e^{z_j}}$，则等价的稳定形式为：

$$\mathcal{L}=-z_t+\log\sum_{j=1}^{C}e^{z_j}.$$

批量训练时通常对所有样本的损失取均值。

In [ ]:
import torch

def cross_entropy_manual(logits, targets):
    """
    logits: [N, C]，未经过 Softmax 的原始分类分数；N 为批大小，C 为类别数。
    targets: [N]，每个元素是范围 [0, C-1] 内的真实类别索引，类型应为 torch.long。
    return: 标量张量 []，即 N 个样本负对数似然损失的均值。
    """
    N = logits.shape[0]  # 标量；logits 的第 0 维对应批次维。
    
    # 1. 计算 LogSoftmax；直接算 log(softmax) 可避免极小概率下的数值下溢。
    #    公式：log_softmax = x - max(x) - log(sum(exp(x - max(x))))。
    # 沿类别维 C 求最大值；keepdim=True 使形状为 [N, 1]，便于广播。
    x_max = logits.max(dim=-1, keepdim=True)[0]  # [N, 1]
    # [N, C] - [N, 1]；每个样本的一行 logits 都减去自己的最大值。
    x_shifted = logits - x_max  # [N, C]
    # 先沿 C 求和：[N, C] -> [N, 1]；log 后形状不变。
    log_sum_exp = torch.log(torch.exp(x_shifted).sum(dim=-1, keepdim=True))  # [N, 1]
    # [N, C] - [N, 1] 的广播操作，得到每个类别对应的对数概率。
    log_softmax = x_shifted - log_sum_exp  # [N, C]
    
    # 2. gather 沿类别维（dim=1）选择每个样本的真实类别。
    # targets: [N] -> [N, 1]；gather 输出 [N, 1]，再压缩为每样本一个值的 [N]。
    target_log_probs = log_softmax.gather(1, targets.unsqueeze(1)).squeeze(1)  # [N]
    
    # 3. 先对 [N] 逐元素取负，再在批次维求均值，输出为 0 维标量张量 []。
    loss = -target_log_probs.mean()
    return loss